# Two-Tower Recommendation Model Training
Simple, piecewise PyTorch Two-Tower training pipeline using **InfoNCE Contrastive Loss** and **Triplet Margin Loss** with explicit negative pairs.

### 1. Imports & Device Setup

In [17]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# Set compute device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### 2. Load Dataset
Load labeled dataset containing positive (`label = 1.0`) and hard negative (`label = 0.0`) match pairs.

In [18]:
# Path resolution
data_path = os.path.join("..", "data", "final_dataset", "labeled_training_dataset.pt")
if not os.path.exists(data_path):
    data_path = os.path.join("src", "data", "final_dataset", "labeled_training_dataset.pt")

dataset = torch.load(data_path, map_location="cpu")

print(f"Loaded dataset from: {data_path}")
print(f"Total samples   : {len(dataset['label']):,}")
print(f"Positive pairs  : {int((dataset['label'] == 1.0).sum().item()):,}")
print(f"Negative pairs  : {int((dataset['label'] == 0.0).sum().item()):,}")

Loaded dataset from: ..\data\final_dataset\labeled_training_dataset.pt
Total samples   : 8,520
Positive pairs  : 1,704
Negative pairs  : 6,816


### 3. Dataset & DataLoaders (Positive & Negative Pairs)
- **`pos_train_loader`**: feeds positive pairs for InfoNCE contrastive alignment.
- **`neg_train_loader`**: feeds negative pairs for Triplet margin repulsion.
- **`val_loader`**: validation positive pairs to evaluate retrieval ranking (MRR, Recall@10, Recall@50).

In [20]:
class MatchingDataset(Dataset):
    def __init__(self, data, indices=None):
        if indices is not None:
            self.data = {k: v[indices] if isinstance(v, torch.Tensor) else [v[i] for i in indices] for k, v in data.items()}
        else:
            self.data = data

    def __len__(self):
        return len(self.data["label"])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}

# 80/20 Train/Val Split
full_ds = MatchingDataset(dataset)
val_len = int(0.2 * len(full_ds))
train_len = len(full_ds) - val_len

train_subset, val_subset = random_split(full_ds, [train_len, val_len], generator=torch.Generator().manual_seed(42))

# Separate positive (label=1) and negative (label=0) training indices
labels = dataset["label"]
pos_train_idx = [train_subset.indices[i] for i in torch.where(labels[train_subset.indices] == 1.0)[0].tolist()]
neg_train_idx = [train_subset.indices[i] for i in torch.where(labels[train_subset.indices] == 0.0)[0].tolist()]
pos_val_idx = [val_subset.indices[i] for i in torch.where(labels[val_subset.indices] == 1.0)[0].tolist()]

# DataLoaders
pos_train_loader = DataLoader(MatchingDataset(dataset, pos_train_idx), batch_size=64, shuffle=True)
neg_train_loader = DataLoader(MatchingDataset(dataset, neg_train_idx), batch_size=64, shuffle=True)
val_loader = DataLoader(MatchingDataset(dataset, pos_val_idx), batch_size=64, shuffle=False)

print(f"Train Positives: {len(pos_train_idx)} | Train Negatives: {len(neg_train_idx)} | Val Positives: {len(pos_val_idx)}")

Train Positives: 1348 | Train Negatives: 5468 | Val Positives: 356


### 4. Two-Tower Neural Network Architecture
- **`PHDTower`**: encodes applicant text ($384$), categorical ($160$), and numerical ($32$) embeddings $\rightarrow 128$-dim.
- **`ProfTower`**: encodes professor text ($384$) and categorical ($160$) embeddings $\rightarrow 128$-dim.

In [21]:
class PHDTower(nn.Module):
    def __init__(self, text_dim=384, cat_dim=160, num_dim=32, hidden_1=512, hidden_2=256, output_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim + cat_dim + num_dim, hidden_1),
            nn.LayerNorm(hidden_1),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_1, hidden_2),
            nn.LayerNorm(hidden_2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_2, output_dim)
        )
    def forward(self, text, cat, num):
        x = torch.cat([text, cat, num], dim=-1)
        return F.normalize(self.net(x), p=2, dim=-1)


class ProfTower(nn.Module):
    def __init__(self, text_dim=384, cat_dim=160, hidden_1=512, hidden_2=256, output_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim + cat_dim, hidden_1),
            nn.LayerNorm(hidden_1),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_1, hidden_2),
            nn.LayerNorm(hidden_2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_2, output_dim)
        )
    def forward(self, text, cat):
        x = torch.cat([text, cat], dim=-1)
        return F.normalize(self.net(x), p=2, dim=-1)


class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.query_tower = PHDTower()
        self.candidate_tower = ProfTower()

    def forward(self, batch):
        q = self.query_tower(batch["phd_text_emb"], batch["phd_cat_emb"], batch["phd_num_emb"])
        c = self.candidate_tower(batch["prof_text_emb"], batch["prof_cat_emb"])
        return q, c

model = TwoTower().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
print(model)

TwoTower(
  (query_tower): PHDTower(
    (net): Sequential(
      (0): Linear(in_features=576, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=512, out_features=256, bias=True)
      (5): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (6): ReLU()
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (candidate_tower): ProfTower(
    (net): Sequential(
      (0): Linear(in_features=544, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=512, out_features=256, bias=True)
      (5): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (6): ReLU()
      (7): Dropout(p=0.2, inplace=False)
      (8): Line

### 5. Loss Functions & Retrieval Metrics

In [22]:
class Loss:
    @staticmethod
    def triplet_margin_loss(q_emb, pos_emb, neg_emb, margin=0.3178):
        """Standard Triplet Margin Loss with explicit negative candidates."""
        return F.triplet_margin_loss(q_emb, pos_emb, neg_emb, margin=margin)

    @staticmethod
    def batch_contrastive_loss(query_embeddings, candidate_embeddings, temperature=0.07, symmetric=True):
        """InfoNCE batch contrastive loss over in-batch negatives."""
        q_norm = F.normalize(query_embeddings, p=2, dim=-1)
        c_norm = F.normalize(candidate_embeddings, p=2, dim=-1)
        
        logits = torch.matmul(q_norm, c_norm.T) / temperature
        labels = torch.arange(logits.size(0), device=logits.device)
        
        forward_loss = F.cross_entropy(logits, labels)
        if not symmetric:
            return forward_loss
        backward_loss = F.cross_entropy(logits.T, labels)
        return 0.5 * (forward_loss + backward_loss)

    @staticmethod
    def calculate_retrieval_metrics(query_embeddings, candidate_embeddings):
        """Calculate Recall@10, Recall@50, and MRR for retrieval ranking."""
        q_norm = F.normalize(query_embeddings, p=2, dim=-1)
        c_norm = F.normalize(candidate_embeddings, p=2, dim=-1)
        sims = torch.matmul(q_norm, c_norm.T).cpu().numpy()
        
        ranks = []
        for i in range(sims.shape[0]):
            order = np.argsort(-sims[i])
            rank = int(np.where(order == i)[0][0]) + 1
            ranks.append(rank)
        ranks = np.array(ranks)
        
        return {
            "recall@10": float(np.mean(ranks <= 10)),
            "recall@50": float(np.mean(ranks <= 50)),
            "mrr": float(np.mean(1.0 / ranks))
        }

### 6. Training & Validation Loop
Train using combined $\mathcal{L} = \mathcal{L}_{\text{triplet}} + 0.5 \cdot \mathcal{L}_{\text{contrastive}}$.

In [23]:
epochs = 15
margin = 0.3178
temperature = 0.07
contrastive_weight = 0.5

print(f"Starting Training for {epochs} Epochs on {device}...")
print("=" * 80)

for epoch in range(1, epochs + 1):
    # ── Training ──
    model.train()
    total_loss, total_triplet, total_contrastive = 0.0, 0.0, 0.0
    num_batches = 0
    
    for pos_batch, neg_batch in zip(pos_train_loader, neg_train_loader):
        optimizer.zero_grad()
        
        pos_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in pos_batch.items()}
        neg_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in neg_batch.items()}
        
        # 1. Positive batch forward pass
        q_pos, c_pos = model(pos_batch)
        
        # 2. Negative batch forward pass
        _, c_neg = model(neg_batch)
        
        # 3. InfoNCE Contrastive Loss (in-batch negatives on positive pairs)
        loss_contrastive = Loss.batch_contrastive_loss(q_pos, c_pos, temperature=temperature)
        
        # 4. Triplet Margin Loss (using explicit negatives)
        min_sz = min(q_pos.size(0), c_neg.size(0))
        loss_triplet = Loss.triplet_margin_loss(q_pos[:min_sz], c_pos[:min_sz], c_neg[:min_sz], margin=margin)
        
        # 5. Combined Loss
        loss = loss_triplet + contrastive_weight * loss_contrastive
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_triplet += loss_triplet.item()
        total_contrastive += loss_contrastive.item()
        num_batches += 1
        
    avg_loss = total_loss / num_batches
    avg_triplet = total_triplet / num_batches
    avg_contrastive = total_contrastive / num_batches
    
    # ── Validation ──
    model.eval()
    val_q, val_c = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            q, c = model(batch)
            val_q.append(q)
            val_c.append(c)
            
    val_metrics = Loss.calculate_retrieval_metrics(torch.cat(val_q, dim=0), torch.cat(val_c, dim=0))
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] "
          f"Loss: {avg_loss:.4f} (Triplet: {avg_triplet:.4f}, NCE: {avg_contrastive:.4f}) | "
          f"Val MRR: {val_metrics['mrr']:.4f} | R@10: {val_metrics['recall@10']:.4f} | R@50: {val_metrics['recall@50']:.4f}")

print("=" * 80)
print("Training Complete!")

Starting Training for 15 Epochs on cpu...
Epoch [01/15] Loss: 1.4600 (Triplet: 0.0896, NCE: 2.7407) | Val MRR: 0.1318 | R@10: 0.3006 | R@50: 0.9551
Epoch [02/15] Loss: 1.1209 (Triplet: 0.0211, NCE: 2.1996) | Val MRR: 0.1196 | R@10: 0.3090 | R@50: 0.9663
Epoch [03/15] Loss: 1.0655 (Triplet: 0.0150, NCE: 2.1011) | Val MRR: 0.1292 | R@10: 0.3455 | R@50: 0.9775
Epoch [04/15] Loss: 1.0316 (Triplet: 0.0165, NCE: 2.0304) | Val MRR: 0.1481 | R@10: 0.3652 | R@50: 0.9551
Epoch [05/15] Loss: 1.0239 (Triplet: 0.0146, NCE: 2.0188) | Val MRR: 0.1411 | R@10: 0.3343 | R@50: 0.9803
Epoch [06/15] Loss: 0.9672 (Triplet: 0.0095, NCE: 1.9155) | Val MRR: 0.1396 | R@10: 0.3539 | R@50: 0.9691
Epoch [07/15] Loss: 0.9659 (Triplet: 0.0164, NCE: 1.8991) | Val MRR: 0.1382 | R@10: 0.3624 | R@50: 0.9719
Epoch [08/15] Loss: 0.9469 (Triplet: 0.0116, NCE: 1.8707) | Val MRR: 0.1356 | R@10: 0.3343 | R@50: 0.9691
Epoch [09/15] Loss: 0.9228 (Triplet: 0.0074, NCE: 1.8308) | Val MRR: 0.1573 | R@10: 0.3708 | R@50: 0.9831
Epoc

### 7. Save Model Weights

In [24]:
ckpt_dir = os.path.join("..", "checkpoints")
if not os.path.exists(ckpt_dir):
    ckpt_dir = os.path.join("src", "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)

ckpt_path = os.path.join(ckpt_dir, "two_tower_checkpoint.pt")
torch.save(model.state_dict(), ckpt_path)
print(f"Model saved to: {ckpt_path}")

Model saved to: ..\checkpoints\two_tower_checkpoint.pt


### 8. Inference: Precompute Unique Professor Catalog & Predict Top-10
Precomputes embeddings over the **unique professor catalog** (deduplicating by `prof_id`) so that each recommended professor is unique.

In [25]:
model.eval()

# 1. Build Unique Professor Candidate Pool (Deduplicated Professor Catalog)
prof_ids_all = dataset["prof_id"]
prof_text_all = dataset["prof_text_emb"]
prof_cat_all = dataset["prof_cat_emb"]

seen = set()
unique_indices = []
for i, pid in enumerate(prof_ids_all.tolist()):
    if pid not in seen:
        seen.add(pid)
        unique_indices.append(i)

unique_idx_tensor = torch.tensor(unique_indices)
unique_prof_ids = [prof_ids_all[i].item() if isinstance(prof_ids_all, torch.Tensor) else prof_ids_all[i] for i in unique_indices]
unique_prof_text = prof_text_all[unique_idx_tensor].to(device)
unique_prof_cat = prof_cat_all[unique_idx_tensor].to(device)

print(f"Total training pairs: {len(prof_ids_all):,} -> Unique Professor Catalog: {len(unique_prof_ids):,} professors")

# 2. Precompute candidate embeddings across unique professors
with torch.no_grad():
    unique_cand_embs = model.candidate_tower(unique_prof_text, unique_prof_cat)  # Shape: (N_unique, 128)
    
    # 3. Query applicant embedding (Sample 0)
    sample_idx = 0
    phd_text = dataset["phd_text_emb"][sample_idx].unsqueeze(0).to(device)
    phd_cat = dataset["phd_cat_emb"][sample_idx].unsqueeze(0).to(device)
    phd_num = dataset["phd_num_emb"][sample_idx].unsqueeze(0).to(device)
    query_emb = model.query_tower(phd_text, phd_cat, phd_num)  # Shape: (1, 128)
    
    # 4. Cosine similarity against unique professor catalog
    sims = torch.matmul(query_emb, unique_cand_embs.T).squeeze(0)
    top_scores, top_indices = torch.topk(sims, k=10)

scholar_id = dataset["scholar_id"][sample_idx].item() if isinstance(dataset["scholar_id"], torch.Tensor) else dataset["scholar_id"][sample_idx]

Total training pairs: 8,520 -> Unique Professor Catalog: 4,096 professors


### 9. Clean Recommendation Detail Viewer
Prints full details for the applicant profile and all top recommended professors clearly without table truncation.

In [27]:
def print_recommendation_details(
    scholar_id,
    top_indices,
    top_scores,
    unique_prof_ids,
    phd_json_path=None,
    prof_csv_path=None
):
    """
    Neatly formats and displays the PhD Scholar profile and each Top-K Recommended Professor cleanly without truncation.
    """
    # Auto-resolve paths
    if phd_json_path is None:
        phd_json_path = os.path.join("..", "data", "processed", "phd", "sop", "gpt_extracted_data.json")
        if not os.path.exists(phd_json_path):
            phd_json_path = os.path.join("src", "data", "processed", "phd", "sop", "gpt_extracted_data.json")
            
    if prof_csv_path is None:
        prof_csv_path = os.path.join("..", "data", "processed", "prof", "extracted_04_prof_data.csv")
        if not os.path.exists(prof_csv_path):
            prof_csv_path = os.path.join("src", "data", "processed", "prof", "extracted_04_prof_data.csv")

    # 1. Load Scholar Profile
    scholar_info = {}
    if os.path.exists(phd_json_path):
        with open(phd_json_path, "r", encoding="utf-8") as f:
            phd_data = json.load(f)
        for item in phd_data:
            if item.get("scholar_id") == scholar_id or item.get("id") == scholar_id:
                scholar_info = item
                break
        if not scholar_info and isinstance(scholar_id, int) and 0 <= scholar_id < len(phd_data):
            scholar_info = phd_data[scholar_id]

    # 2. Load Professor CSV
    prof_df = pd.read_csv(prof_csv_path) if os.path.exists(prof_csv_path) else pd.DataFrame()

    # ── Display Scholar Profile ──
    print("=" * 85)
    print(f" SCHOLAR PROFILE [ID: {scholar_id}]")
    print(f"   Name               : {scholar_info.get('name', 'N/A')}")
    print(f"   University         : {scholar_info.get('university', 'N/A')}")
    print(f"   Department         : {scholar_info.get('department', 'N/A')}")
    interests = scholar_info.get("research_interests", [])
    if isinstance(interests, list) and interests:
        print(f"   Research Interests : {', '.join(interests)}")
    print("=" * 85)

    # ── Display Professor Cards ──
    print(f"\n TOP-{len(top_indices)} RECOMMENDED PROFESSORS:")
    print("=" * 85)

    indices_list = top_indices.cpu().tolist() if isinstance(top_indices, torch.Tensor) else top_indices
    scores_list = top_scores.cpu().tolist() if isinstance(top_scores, torch.Tensor) else top_scores

    for rank, (idx, score) in enumerate(zip(indices_list, scores_list), start=1):
        prof_id = unique_prof_ids[idx]
        row = prof_df[prof_df["id"] == prof_id] if not prof_df.empty else pd.DataFrame()

        p_name = row.iloc[0].get("name", "N/A") if not row.empty else "N/A"
        p_dept = row.iloc[0].get("department", "N/A") if not row.empty else "N/A"
        p_title = row.iloc[0].get("title", "N/A") if not row.empty else "N/A"
        p_exp = row.iloc[0].get("expertise", "N/A") if not row.empty else "N/A"

        print(f"\n[Rank {rank:02d}] ── Professor ID: {prof_id} | Similarity Match: {score:.4f}")
        print(f"  • Name       : {p_name}")
        print(f"  • Department : {p_dept}")
        print(f"  • Key Title  : {p_title}")
        print(f"  • Expertise  : {p_exp}")
        print("-" * 85)

# Execute and view full details
print_recommendation_details(
    scholar_id=scholar_id,
    top_indices=top_indices,
    top_scores=top_scores,
    unique_prof_ids=unique_prof_ids
)

 SCHOLAR PROFILE [ID: 150]
   Name               : Maya L. Patel
   University         : Stanford University
   Department         : Electrical and Computer Engineering
   Research Interests : Autonomous navigation for aerial robotics, Multi‑agent reinforcement learning, Safety‑critical AI for UAV swarms, Sensor fusion and visual‑inertial odometry, Edge AI and low‑power inference, Explainable decision‑making in autonomous systems, Human‑robot interaction in mixed‑initiative missions

 TOP-10 RECOMMENDED PROFESSORS:

[Rank 01] ── Professor ID: 3970 | Similarity Match: 0.8800
  • Name       : Sudeep Tanwar
  • Department : Institute of Technology
  • Key Title  : Deployment of Unmanned Aerial Vehicles in Next-Generation Wireless Communication Network Using Multi-Agent Reinforcement Learning
  • Expertise  : Blockchain Technology, Wireless Networking, Cyber-Physical Systems, Internet of Things, ML/DL in Communication
------------------------------------------------------------------------